In [1]:
import pandas as pd
import numpy as np

### Data Loading and Exploration

In [2]:
def load_arff_to_dataframe(file_path):
    """
    Load ARFF file and convert it to a pandas DataFrame
    """
    try:
        with open(file_path, 'r') as f:
            lines = f.readlines()
        
        # Find attribute definitions and data section
        attributes = []
        data_start_idx = 0
        
        for i, line in enumerate(lines):
            line = line.strip()
            if line.startswith('@attribute'):
                # Extract attribute name (remove quotes if present)
                parts = line.split()
                attr_name = parts[1].replace("'", "").replace('"', '')
                attributes.append(attr_name)
            elif line.startswith('@data'):
                data_start_idx = i + 1
                break
        
        # Extract data rows
        data_rows = []
        for i in range(data_start_idx, len(lines)):
            line = lines[i].strip()
            if line and not line.startswith('%'):  # Skip empty lines and comments
                # Split by comma and clean each value
                row = [val.strip() if val.strip() != '?' else np.nan for val in line.split(',')]
                data_rows.append(row)
        
        # Ensure consistent row lengths - trim to match number of attributes
        for i, row in enumerate(data_rows):
            if len(row) > len(attributes):
                data_rows[i] = row[:len(attributes)]  # Trim extra columns
            elif len(row) < len(attributes):
                data_rows[i].extend([np.nan] * (len(attributes) - len(row)))  # Pad with NaN
        
        # Create DataFrame
        df = pd.DataFrame(data_rows, columns=attributes)
        
        # Convert numeric columns to appropriate types
        numeric_cols = ['age', 'bp', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wbcc', 'rbcc']
        for col in numeric_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')
        
        return df
        
    except Exception as e:
        print(f"Error loading ARFF file: {e}")
        return None



In [3]:
# Load the chronic kidney disease dataset
file_path = 'chronic_kidney_disease.arff'
df = load_arff_to_dataframe(file_path)

df.head()

,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wbcc,rbcc,htn,dm,cad,appet,pe,ane,class
0,48.0,80.0,1.020,1,0,NaN,normal,notpresent,notpresent,121.0,...,44.0,7800.0,5.2,yes,yes,no,good,no,no,ckd
1,7.0,50.0,1.020,4,0,NaN,normal,notpresent,notpresent,NaN,...,38.0,6000.0,NaN,no,no,no,good,no,no,ckd
2,62.0,80.0,1.010,2,3,normal,normal,notpresent,notpresent,423.0,...,31.0,7500.0,NaN,no,yes,no,poor,no,yes,ckd
3,48.0,70.0,1.005,4,0,normal,abnormal,present,notpresent,117.0,...,32.0,6700.0,3.9,yes,no,no,poor,yes,yes,ckd
4,51.0,80.0,1.010,2,0,normal,normal,notpresent,notpresent,106.0,...,35.0,7300.0,4.6,no,no,no,good,no,no,ckd


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 25 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   age     391 non-null    float64
 1   bp      388 non-null    float64
 2   sg      353 non-null    object 
 3   al      354 non-null    object 
 4   su      351 non-null    object 
 5   rbc     248 non-null    object 
 6   pc      335 non-null    object 
 7   pcc     396 non-null    object 
 8   ba      396 non-null    object 
 9   bgr     356 non-null    float64
 10  bu      381 non-null    float64
 11  sc      383 non-null    float64
 12  sod     313 non-null    float64
 13  pot     312 non-null    float64
 14  hemo    348 non-null    float64
 15  pcv     329 non-null    float64
 16  wbcc    294 non-null    float64
 17  rbcc    269 non-null    float64
 18  htn     398 non-null    object 
 19  dm      398 non-null    object 
 20  cad     398 non-null    object 
 21  appet   399 non-null    object 
 22  pe

In [12]:
print(f"\nMissing values:")
df.isnull().sum()


Missing values:


age        9
bp        12
sg        47
al        46
su        49
rbc      152
pc        65
pcc        4
ba         4
bgr       44
bu        19
sc        17
sod       87
pot       88
hemo      52
pcv       71
wbcc     106
rbcc     131
htn        2
dm         2
cad        2
appet      1
pe         1
ane        1
class      0
dtype: int64

Separate numerical and categorical features

In [9]:
# check all datatypes of dataframe 
df.dtypes

age      float64
bp       float64
sg        object
al        object
su        object
rbc       object
pc        object
pcc       object
ba        object
bgr      float64
bu       float64
sc       float64
sod      float64
pot      float64
hemo     float64
pcv      float64
wbcc     float64
rbcc     float64
htn       object
dm        object
cad       object
appet     object
pe        object
ane       object
class     object
dtype: object

In [10]:
numertic_cols = df.select_dtypes(include=['float64', 'int64']).columns
catergorical_cols = df.select_dtypes(include=['object']).columns

print(f"numertic_cols: {numertic_cols}")
print(f"catergorical_cols: {catergorical_cols}")

numertic_cols: Index(['age', 'bp', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wbcc',
       'rbcc'],
      dtype='object')
catergorical_cols: Index(['sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'htn', 'dm', 'cad', 'appet',
       'pe', 'ane', 'class'],
      dtype='object')


### Preprocess

In [14]:
df.sg.mode()

0    1.020
Name: sg, dtype: object

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Encode categorical columns
df_processed = df.copy()
for col in catergorical_cols:
    
    if col in df_processed.columns:
        # filling missing values with mode
        

        
    
    

NameError: name 'catergorical_cols' is not defined

25